In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from wiki_forecast.features import FEATURES_WITH_CORR, build_dataset, to_log_wide

df = pd.read_parquet("../data/raw/pageviews.parquet")
log_wide = to_log_wide(df)
log_wide.shape

(4146, 553)

In [2]:
ORIGIN_STEP = 14          # a forecast origin every 14 days
HORIZONS = range(1, 29)   # 1 to 28 days ahead

origins = pd.date_range("2018-01-01", log_wide.index[-1], freq=f"{ORIGIN_STEP}D")
data = build_dataset(log_wide, origins, HORIZONS)

print(f"{len(data):,} rows, {data.memory_usage(deep=True).sum() / 1e6:.0f} MB")
data.head()

3,449,712 rows, 436 MB


,article,origin,level,lag_0,lag_1,lag_2,lag_3,lag_6,lag_13,lag_27,...,std_28,season_corr,horizon,season_1y,season_2y,target,target_date,dow,doy,month
0,1923_(TV_series),2022-09-19,7.280766,0.038436,0.135011,0.031787,-0.053830,0.008162,0.207527,NaN,...,NaN,NaN,1,NaN,NaN,-0.058930,2022-09-20,1,263,9
1,1923_(TV_series),2022-10-03,7.433511,0.166891,0.109762,-0.017133,0.065358,-0.105731,-0.211675,0.054782,...,0.110936,NaN,1,NaN,NaN,0.068123,2022-10-04,1,277,10
2,1923_(TV_series),2022-10-17,7.603207,0.135717,0.161938,-0.029675,0.153845,-0.157205,-0.101572,-0.381371,...,0.150616,NaN,1,NaN,NaN,0.068620,2022-10-18,1,291,10
3,1923_(TV_series),2022-10-31,7.873560,-0.044726,-0.024627,-0.021121,0.470945,-0.357127,-0.201734,-0.371926,...,0.227517,NaN,1,NaN,NaN,-0.122515,2022-11-01,1,305,11
4,1923_(TV_series),2022-11-14,8.838109,1.425811,0.763056,0.595775,-0.250644,-0.783587,-1.087064,-1.166282,...,0.681307,NaN,1,NaN,NaN,0.861915,2022-11-15,1,319,11


In [3]:
cols = ["origin", "horizon", "target_date", "level", "lag_0", "season_1y", "target"]
okt = data[(data["article"] == "Oktoberfest") & (data["origin"] == pd.Timestamp("2025-09-01"))]
okt[cols]

,origin,horizon,target_date,level,lag_0,season_1y,target
82686,2025-09-01,1,2025-09-02,7.870528,0.167662,0.073716,0.150729
205890,2025-09-01,2,2025-09-03,7.870528,0.167662,0.122880,0.163455
329094,2025-09-01,3,2025-09-04,7.870528,0.167662,0.157837,0.200691
452298,2025-09-01,4,2025-09-05,7.870528,0.167662,0.174674,0.305020
575502,2025-09-01,5,2025-09-06,7.870528,0.167662,0.198852,0.378001
698706,2025-09-01,6,2025-09-07,7.870528,0.167662,0.238339,0.365098
821910,2025-09-01,7,2025-09-08,7.870528,0.167662,0.283850,0.290847
945114,2025-09-01,8,2025-09-09,7.870528,0.167662,0.325002,0.296257
1068318,2025-09-01,9,2025-09-10,7.870528,0.167662,0.377557,0.199440
1191522,2025-09-01,10,2025-09-11,7.870528,0.167662,0.413150,0.111206


In [4]:
origin = pd.Timestamp("2024-09-01")

full = build_dataset(log_wide, [origin], HORIZONS)
truncated = build_dataset(log_wide.loc[:origin], [origin], HORIZONS)

a = full.set_index(["article", "target_date"])[FEATURES_WITH_CORR]
b = truncated.set_index(["article", "target_date"])[FEATURES_WITH_CORR]

print("Same NaN pattern:", (a.isna() == b.isna()).all().all())
print("Max difference:  ", (a - b).abs().max().max())
print("Targets in truncated version:", truncated["target"].notna().sum())

Same NaN pattern: True
Max difference:   0.0
Targets in truncated version: 0
